**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/peron-2015"], check=True)
    os.chdir("/content/encore/entries/peron-2015")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Peron et al. (2015) from CRCNS ssc-1

Paper. Peron SP, Freeman J, Iyer V, Guo C, Svoboda K (2015). A cellular resolution map of barrel
cortex activity during tactile behavior. *Neuron* 86:783-799.
[doi:10.1016/j.neuron.2015.03.027](https://doi.org/10.1016/j.neuron.2015.03.027). The claims file
(`claims/peron-2015.yaml`) was written from the version of record with its Supplemental Information.

Data set. [CRCNS ssc-1](https://doi.org/10.6080/K0TB14TN), read anonymously from the public S3
bucket `crcnsarchive` under `ssc-1/`. CRCNS data sets are not versioned (docs/archives.md, "CRCNS"),
so the version recorded is `unversioned`; the paper cites the data set's DOI and no version, and
the bucket holds one copy (no `live/` or `old/` folder). A later change is detectable from the MD5
sums of the files read, taken from the data set's `checksums.md5` and checked against the bytes
read:

| File | MD5 |
|---|---|
| an197522_2013_03_08/an197522_2013_03_08_session.mat | 1d4bd20b7bd82a3bb41b0655d45998c0 |
| an197522_2013_03_10/an197522_2013_03_10_session.mat | f0dfe39427828d0b1f77f0bed35412d6 |
| an229717_2013_12_01/an229717_2013_12_01_sesssion.mat | 36048ccc26b241ef7c35734d35eae61a |
| an229719_2013_12_02/an229719_2013_12_02_session.mat | 52091811e35b8fe11a6efac6d23f4ac7 |
| an229719_2013_12_05/an229719_2013_12_05_session.mat | f9ceda31c2a7446541bb10b6f8a5d355 |

What the data set holds. Five behavioural imaging sessions from three of the paper's eight
volume-imaging mice (an197522, an229717, an229719, all Emx1-Cre x LSL-H2B-mCherry), each as one
MATLAB structure with trial types and timing, pole position, licks, rewards, whisker angle and
curvature change at 500 Hz, touch times, and dF/F per ROI with the ROI masks. It also holds the
registered imaging TIFFs and the whisker videos of the same sessions. It holds no calcium events,
no raw or neuropil fluorescence, no layer, cell-class or column labels, no multi-session `vol_`
files, and none of the learning, L4 axon, anesthetized, L5 soma and dendrite, high-resolution stack
or training data. The data check below shows this.

Revision after the first verification. Added: the contents of every identifier and description
field of the dF/F series (no layer, class or column label in any of them), the trial numbering of
each session, and, for the whisking amplitude, the readings without tracking-error samples, after
unwrapping angles past 90 degrees, without the filter's edge effects, and with per-mouse medians.
After the second, reading (a) without tracking errors that last several samples.

Claims. Two of the 43 claims are tested; the rest are blocked. The claims file's shared method
M6 says that "if [the data set] carries none [column and layer labels], the claims that need them
are blocked, as no rule here could redraw them".

| Claim | Status |
|---|---|
| results-trials-per-session | tested |
| results-whisking-amplitude | tested |
| table-s1-roi-counts | blocked: most of the sessions are not in the data set (counts shown in the data check) |
| fig2-core-counts | blocked: no cell-class labels (counts shown in the data check) |
| results-gaba-fraction, fig4d-gaba-reward-cue, fig2e-fraction-imaged, fig1e-days-to-criterion, fig1g-early-stereotypy, supp-trim-all-chance, figs4-dendrite-soma | blocked: data not in the data set |
| fig3a, fig3b, fig3c, fig3d (four claims), discussion-principal-exclusive-classes, fig4d-sample-epoch-principal, fig7 (three claims), fig8b, fig8de, results-di-magnitude-similar, discussion-correlation-more-touch | blocked: no layer, cell-class or column labels, and no calcium events |
| figs3d-neuropil-correction | blocked: no L2/3 labels; raw ROI and neuropil fluorescence could be rebuilt from the registered TIFFs (146 GB of imaging archives, cell 3) and the ROI masks, but the claim would still need the L2/3 labels |
| fig4-neuropil-sample-principal, figs5-l4-neuropil-correlation, fig8e-l4-neuropil-direction-map | blocked: no neuropil or L4 axon ROIs |
| fig6 (seven claims), figs7-direction-stable | blocked: no imaging during learning |
| fig8h, fig8i, fig8jk | blocked: no anesthetized mice |
| methods-event-detection | blocked: outside data, method not given |

In [ ]:
import time
import resource

import numpy as np
import pandas as pd

import ssc1

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
keys = ssc1.list_bucket()
sessions = ssc1.load_all()
print(f"loaded {len(sessions)} session files in {time.time() - started:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Data check

Each check of the description against the values is stated under its output. The description
is the data set's `docs/crcns_ssc-1_data_description.pdf` (version 0.8, also under `paper/`).

In [ ]:
# Files in the bucket, grouped by kind.
df = pd.DataFrame(keys, columns=["key", "bytes"])
df = df[~df.key.str.endswith("/")]
df["folder"] = df.key.str.split("/").str[1]
df["kind"] = np.select(
    [df.key.str.endswith(".mat"), df.key.str.contains("imaging_data/"), df.key.str.contains("whisking_data")],
    ["session .mat", "imaging tar.gz", "whisker video tar.gz"], "other")
print(f"{len(df)} files, {df.bytes.sum() / 1e9:.0f} GB")
print(df.groupby(["folder", "kind"]).agg(files=("key", "size"), GB=("bytes", lambda b: round(b.sum() / 1e9, 1))).to_string())
print("\nfiles with 'vol_' in the name (multi-session files the description names):",
      int(df.key.str.contains("vol_").sum()))
print("files under ssc-1/live/ or ssc-1/old/:", int(df.key.str.contains("/live/|/old/").sum()))
print("\nMD5 of each session file read against checksums.md5:")
for s in sessions:
    print(f"  {s['name']}: {'match' if s['md5'] == ssc1.MD5[s['name']] else 'MISMATCH ' + s['md5']}")

The bucket holds five session folders, each with one session `.mat` file, the registered imaging
TIFFs (one `tar.gz` per imaging plane) and the whisker videos. The description says files named
`vol_xxx` hold multi-session data; there are none. Each session file was checked against the
data set's MD5 sums above.

In [ ]:
# Everything one session structure holds (the first file), field by field.
s0 = sessions[2]
print("metaDataHash:", s0["meta"])
print("time units:", s0["time_units"], "; trialTimeUnit:", s0["trial_time_unit"])
print("time series:", s0["timeseries_keys"])
print("event series:", s0["event_series"], "; event types:", s0["event_types"])
print("whisker variables:", s0["whisker_summary"]["names"])
print("fields of a dF/F time series:", s0["subvolumes"][0]["fields"])
print("touch series:", list(s0["touches"]))
print("trial table fields used:", list(s0["trials"]))

The fields match the description's list, with two differences: the structure is called
`eventSeriesArrayHash` (the description says `eventSeriesHash`), and the time series hold only
`whiskerVars` and one `dffTSA` per subvolume. The description lists "raw fluorescence for each ROI"
and "algorithmically extracted calcium events for each ROI" among the processed data; neither is
in any session file. The imaging archives (one per plane) hold registered TIFFs; one of them,
listed in full while preparing this notebook (not run here), held 45 TIFFs and nothing else.

In [ ]:
# What the identifier and description fields of every dF/F series hold, to see whether any of
# them carries a layer, cell-class or column label: the series descriptions, the patterns of
# idStrs (digits replaced by <n>), the entries of idStrsDetailed, the keys of the description
# hash, the fields of the ROI definitions and imaging planes, and how the ROI ids relate to planes.
import re

rows = []
for s in sessions:
    for v in s["subvolumes"]:
        thousands = {int(p): sorted({int(i) // 1000 for i in v["ids"][v["plane"] == p]}) for p in np.unique(v["plane"])}
        rows.append({"session": s["name"], "key": v["key"], "descr": v["descr"],
                     "idStrs patterns": sorted({re.sub(r"\d+", "<n>", x) for x in v["id_strs"]}),
                     "idStrs name the id": all(x.endswith(str(i)) for x, i in zip(v["id_strs"], v["ids"])),
                     "idStrsDetailed entries": len(v["id_strs_detailed"]),
                     "ROI id // 1000 per plane": thousands})
lab = pd.DataFrame(rows)
print(lab.to_string(index=False))
print("\ndistinct idStrs patterns over all series:", sorted({x for p in lab["idStrs patterns"] for x in p}))
print("idStrsDetailed entries over all series:", int(lab["idStrsDetailed entries"].sum()))
print("descrHash keys:", sorted({tuple(v["descr_keys"]) for s in sessions for v in s["subvolumes"]}))
print("fields of an ROI set:", sorted({tuple(v["roi_array_fields"]) for s in sessions for v in s["subvolumes"]}))
print("fields of an ROI:", sorted({tuple(v["roi_fields"]) for s in sessions for v in s["subvolumes"]}))
print("fields of an imaging plane:", sorted({tuple(v["plane_fields"]) for s in sessions for v in s["subvolumes"]}))
print("metaDataHash keys of each session:", sorted({tuple(s["meta"]) for s in sessions}))


Every dF/F series is described only as "dF/F for ROIs in subvolume <n>"; every `idStrs` entry
is "dF/F Ca TS for <id>", which repeats the ROI id; `idStrsDetailed` is empty in every series. The
ROI definitions hold an id, the outline corners and the pixel indices, and the planes hold ids and
source files. The thousands part of an ROI id (id // 1000) is one number per plane, rising through
the session's subvolumes (printed per plane above); it numbers planes and says nothing of depth,
layer, class or column. The session metadata hold species, strain, genotype, animal, sex, date,
time and experimenters. So no field carries the labels
that the claims of shared method M6 need, which confirms the blocks below.

In [ ]:
rows = []
for s in sessions:
    tr = s["trials"]
    lab = pd.Series(tr["label"]).value_counts()
    rows.append({"session": s["name"], "mouse": s["meta"]["animalID"],
                 "genotype": s["meta"]["animalGeneModification1"] + " x " + s["meta"]["animalGeneModification2"],
                 "date": s["meta"]["dateOfExperiment"], "whisker": s["whisker_summary"]["names"][0].split()[-1].upper(),
                 "trials": len(tr["id"]), "trial ids unique": len(np.unique(tr["id"])) == len(tr["id"]),
                 **{k: int(lab.get(k, 0)) for k in ["HitL", "HitR", "ErrL", "ErrR", "NoLickL", "NoLickR"]},
                 "one label per trial": bool(np.all(tr["n_labels"] == 1)),
                 "subvolumes": len(s["subvolumes"]), "ROIs": int(sum(len(v["ids"]) for v in s["subvolumes"]))})
info = pd.DataFrame(rows)
print(info.to_string(index=False))
print("\nTable S1 (claims file): an197522 D2, an229717 D3, an229719 D2.")

Three mice, all Emx1-Cre x LSL-H2B-mCherry, as Table S1 lists them, and the whisker named in each
session's whisker variables is the spared whisker Table S1 gives (D2, D3, D2). Every trial has
exactly one of the six labels. The other five volume-imaging mice of the paper, including the
Gad2 mouse an194181 with labelled GABAergic neurons, are not in the data set.

In [ ]:
# Does the trial table cover the whole behavioural session or only its imaged part? Trial ids,
# their first and last values, gaps in numbering, the trials that have imaging, and the intervals
# between trial starts.
rows = []
for s in sessions:
    tr = s["trials"]
    ids, st = tr["id"], tr["start"]
    imaged = np.unique(np.concatenate([v["trials"] for v in s["subvolumes"]]))
    gaps = np.diff(st) / 1000
    k = int(np.argmax(gaps))
    rows.append({"session": s["name"], "trials": len(ids), "first id": int(ids[0]), "last id": int(ids[-1]),
                 "ids contiguous": bool(np.all(np.diff(ids) == 1)),
                 "imaged ids first-last": [int(imaged.min()), int(imaged.max())], "trials with imaging": len(imaged),
                 "imaged ids in table": bool(np.isin(imaged, ids).all()),
                 "median interval s": round(float(np.median(gaps)), 1),
                 "longest interval s (after id)": [round(float(gaps[k])), int(ids[k])],
                 "span min": round(float((st[-1] - st[0]) / 60000), 1)})
print(pd.DataFrame(rows).to_string(index=False))


Trial ids run from 1 without gaps in every session, as the behavioural controller numbers trials
from the start of a session, and the trials with imaging are a subset of them. So the trial table
covers the behavioural session from its first trial, not only its imaged part. This is consistent
with the data, not shown: whether trials ran after the last one in the table cannot be seen. In every
session the longest interval between trial starts follows trial 1 (12.5 min to 2.4 h, against a
median of 10 to 11 s), so the first trial is separated from the rest by a pause; dropping it would
change each count by one.

In [ ]:
# Trial timing: sample epoch (pole in reach), reward cue, pole positions by correct side.
rows = []
for s in sessions:
    tr = s["trials"]
    dur = tr["pole_out"] - tr["pole_in"]
    cue = tr["cue"] - tr["pole_in"]
    left = np.isin(tr["label"], ["HitL", "ErrL", "NoLickL"])
    right = np.isin(tr["label"], ["HitR", "ErrR", "NoLickR"])
    pos = tr["pole_position"]
    rows.append({"session": s["name"],
                 "pole-in duration ms (min/median/max)": np.round(np.nanpercentile(dur, [0, 50, 100])).astype(int).tolist(),
                 "cue after pole-in ms (median)": int(np.nanmedian(cue)),
                 "trials without pole-in": int(np.isnan(tr["pole_in"]).sum()),
                 "trials without cue": int(np.isnan(tr["cue"]).sum()),
                 "position L (min-max, steps)": [int(np.nanmin(pos[left & (pos > 0)])), int(np.nanmax(pos[left]))],
                 "position R (min-max, steps)": [int(np.nanmin(pos[right & (pos > 0)])), int(np.nanmax(pos[right]))],
                 "positions 0": int((pos == 0).sum())})
print(pd.DataFrame(rows).to_string(index=False))

The sample epoch (pole in reach) lasts 0.84 to 1.18 s in the median, near the paper's 1 s, and
the reward cue comes about 1.9 to 2.1 s after the pole comes into reach, which is the 1 s sample
plus 1 s delay of the paper's trial structure. The two correct sides use separate, non-overlapping
ranges of pole position in every session, as the task requires; which range is proximal is not
stated in the files. A few trials have position 0 and no pole-in time (the first trial of a
session, among others).

In [ ]:
# Whisker variables: units, sampling and ranges.
rows = []
for s in sessions:
    w = s["whisker_summary"]
    rows.append({"session": s["name"], "time unit": s["time_units"][w["time_unit"]], "dt ms": w["dt_ms"],
                 "samples": w["n"], "angle min/max (deg)": np.round(w["angle_range"], 1).tolist(),
                 "angle 1/50/99 pct": np.round(w["angle_pct"], 1).tolist(),
                 "|angle| > 85 deg": f"{w['angle_abs_over_85']:.5f}", "angle NaN": f"{w['angle_nan']:.5f}",
                 "kappa 1/50/99 pct": np.round(w["kappa_pct"], 5).tolist(),
                 "valid whisker trials": len(s["valid_whisker_trials"])})
print(pd.DataFrame(rows).to_string(index=False))

Whisker samples are 2 ms apart (500 Hz, as described), in milliseconds. Angles lie mostly between
about -40 and +60 degrees, with rare samples at the limits of +/-90 degrees (at most 0.06% of
samples), which look like tracking errors. Curvature change is centred on 0, as the paper's
"zero for non-touch periods" implies; its units are not stated in the files or the description.

In [ ]:
# Touches against the trial table: share of touch onsets within the pole-in-reach window.
rows = []
for s in sessions:
    tr = s["trials"]
    for name, v in s["touches"].items():
        k = np.searchsorted(tr["id"], v["trial"])
        inside = (v["start"] >= tr["pole_in"][k] - 50) & (v["start"] <= tr["pole_out"][k] + 500)
        rows.append({"session": s["name"], "series": name, "touches": len(v["start"]),
                     "within pole window (-50 ms, +500 ms)": f"{inside.mean():.3f}"})
print(pd.DataFrame(rows).to_string(index=False))

Touch times come from the whisker video and the pole-in-reach times from the behavioural
controller; 96 to 100% of touch onsets fall within the pole window (allowing 0.5 s after the
pole starts to withdraw), so the whisker and trial clocks agree.

In [ ]:
# Imaging: one row per subvolume.
rows = []
for s in sessions:
    for v in s["subvolumes"]:
        rows.append({"session": s["name"], "key": v["key"], "raw folder": v["fov"],
                     "ROIs": len(v["ids"]), "ROIs per plane": v["plane_counts"],
                     "frame ms": round(v["frame_ms"], 2), "Hz": round(1000 / v["frame_ms"], 2),
                     "trials": len(v["trials"]), "valid Ca trials": v["n_valid_trials"],
                     "NaN": round(v["nan_fraction"], 3),
                     "dF/F min/median/max": np.round(v["dff_range"], 3).tolist()})
im = pd.DataFrame(rows)
print(im.to_string(index=False))
r7 = im[im.Hz.between(6.5, 7.5)]
print(f"\nsubvolumes at about 7 Hz: {len(r7)} of {len(im)}; trials per subvolume (7 Hz): "
      f"{r7.trials.mean():.1f} +/- {r7.trials.std():.1f} (paper: 32 +/- 7)")

Of 34 subvolumes, 33 are imaged at 7.0 Hz, as described, for 28 to 45 trials each (the paper says
32 +/- 7). One, `dffTSA 10` of an197522 on 2013-03-08 (raw folder `fov_04001`), is a single plane
at 29 Hz with 202 ROIs; the description does not mention it. The description says each subvolume
has three planes; many have ROIs on only one or two of the three (for example 540 ROIs on one plane
in `fov_03004`). dF/F is unitless, from about -1 to 14, with 0 to 15% missing values per subvolume.

In [ ]:
# Recordings deposited more than once: identical dF/F matrices, and ROI sets shared between sessions.
hashes = {}
for s in sessions:
    for v in s["subvolumes"]:
        hashes.setdefault(v["hash"], []).append((s["name"], v["key"]))
dups = [h for h in hashes.values() if len(h) > 1]
print("identical dF/F matrices:", dups if dups else "none")
for m in ["an197522", "an229719"]:
    ss = [s for s in sessions if s["meta"]["animalID"] == m]
    a = {v["fov"][:6]: v for v in ss[0]["subvolumes"]}
    b = {v["fov"][:6]: v for v in ss[1]["subvolumes"]}
    for k in sorted(set(a) & set(b)):
        same = np.array_equal(a[k]["ids"], b[k]["ids"])
        print(f"{m}: subvolume {k} in {ss[0]['name']} ({a[k]['n_frames']} frames) and {ss[1]['name']} "
              f"({b[k]['n_frames']} frames); same ROI ids: {same}; same dF/F: {a[k]['hash'] == b[k]['hash']}")
    if not set(a) & set(b):
        print(f"{m}: no subvolume in both sessions")

No recording is deposited twice. In an229719 the subvolume of raw folder `fov_06` is imaged in
both sessions, with the same 1,332 ROI ids and different recordings (different frame counts and
values). This matches the paper's design (each subvolume imaged over several sessions, ROIs kept
across sessions), so these ROIs are counted once per mouse below; that the ids name the same
neurons in both sessions is taken from the description ("ROI ids are consistent"), not shown.

In [ ]:
# Imaging clock against the trial clock: population dF/F after minus before each trial's first
# touch, with the true touch times and with the times shifted by -3 s and +3 s.
rows = []
for s in sessions:
    for v in s["subvolumes"]:
        rows.append({"session": s["name"], "folder": v["fov"],
                     "true": v["touch"][0.0][0], "shift -3 s": v["touch"][-3000.0][0],
                     "shift +3 s": v["touch"][3000.0][0], "trials": v["touch"][0.0][1]})
ck = pd.DataFrame(rows)
ck["true highest"] = (ck["true"] > ck["shift -3 s"]) & (ck["true"] > ck["shift +3 s"])
print(ck.round(4).to_string(index=False))
print("\nper session: subvolumes where the true alignment gives the largest response, and median values")
print(ck.groupby("session").agg(n=("true", "size"), true_highest=("true highest", "sum"),
                                true=("true", "median"), minus3=("shift -3 s", "median"),
                                plus3=("shift +3 s", "median")).round(4).to_string())

The touch response is a contrast that a clock offset would remove. In an229717 (4 of 4 subvolumes)
and an229719 (10 of 11) the true alignment gives the largest response, with session medians of
0.009 to 0.046 dF/F against -0.010 to 0.005 when shifted, so dF/F shares the trial clock there. In
an197522 on 2013-03-08 it does in 7 of 10 subvolumes, with smaller responses (median 0.014 against
-0.002 and 0.010). On 2013-03-10 it
does in only 3 of 9 subvolumes, and the median response is no larger than with the shift, so for
that session the shared clock could not be established. No tested claim uses dF/F.

In [ ]:
# ROI counts per mouse (claim table-s1-roi-counts is blocked; the counts are shown here).
table_s1 = {"an171923": 11031, "an194181": 9637, "an194672": 13788, "an197522": 16200,
            "an198503": 8288, "an229716": 12293, "an229717": 19012, "an229719": 18125}
rows = []
for m, ref in table_s1.items():
    ss = [s for s in sessions if s["meta"]["animalID"] == m]
    ids = set()
    ids7 = set()
    for s in ss:
        for v in s["subvolumes"]:
            ids |= set(v["ids"].tolist())
            if 1000 / v["frame_ms"] < 10:
                ids7 |= set(v["ids"].tolist())
    rows.append({"mouse": m, "sessions in data set": len(ss), "distinct ROIs": len(ids) if ss else None,
                 "distinct ROIs, 7 Hz subvolumes": len(ids7) if ss else None, "Table S1": ref,
                 "fraction of Table S1": round(len(ids) / ref, 2) if ss else None})
rc = pd.DataFrame(rows)
print(rc.to_string(index=False))
print(f"\ntotal distinct ROIs in the data set: {int(rc['distinct ROIs'].sum())} "
      f"(Table S1, eight mice: {sum(table_s1.values())})")

The data set holds 3 of the 8 volume-imaging mice, and for these 72%, 20% and 63% of the ROIs
Table S1 gives (11,649 of 16,200, 3,771 of 19,012 and 11,341 of 18,125); 26,761 ROIs in all,
against 108,374 for the eight mice. The paper imaged 16 subvolumes per mouse over about
four sessions each; the data set holds one or two sessions per mouse. The per-mouse counts of
Table S1 rest on sessions that are not deposited, so table-s1-roi-counts and fig2-core-counts are
blocked.

The data set's metadata hold no layer, cell-class (excitatory, GABAergic, L5 apical dendrite) or
column label for any ROI, no depth for any plane, and no position of one subvolume relative to
another; the ROI masks give positions only within each 512 x 512 plane.

## results-trials-per-session

> Mice performed 245 ± 71 trials per session during imaging.

Criterion: the number of behavioural trials per volume-imaging session; readings are the mean
over all sessions pooled and the mean of per-mouse means; reproduced when one reading is within
245 ± 71.

In [ ]:
tps = info[["session", "mouse", "trials"]].copy()
# Variant: trials with imaging, the union over subvolumes of their valid calcium trials.
tps["trials with imaging"] = [len(np.unique(np.concatenate([v["trials"] for v in s["subvolumes"]]))) for s in sessions]
print(tps.to_string(index=False))
pooled = tps.trials.mean()
per_mouse = tps.groupby("mouse").trials.mean()
print("\nper-mouse means:", per_mouse.round(1).to_dict())
lo, hi = 245 - 71, 245 + 71
print(f"(1) pooled over {len(tps)} sessions: {pooled:.1f} +/- {tps.trials.std():.1f} (SD); within [{lo}, {hi}]: {lo <= pooled <= hi}")
print(f"(2) mean of {len(per_mouse)} per-mouse means: {per_mouse.mean():.1f} +/- {per_mouse.std():.1f}; within: {lo <= per_mouse.mean() <= hi}")
v = tps["trials with imaging"]
print(f"variant, trials with imaging: pooled {v.mean():.1f} +/- {v.std():.1f}; per-mouse mean {tps.groupby('mouse')['trials with imaging'].mean().mean():.1f}")
print(f"range over sessions: {tps.trials.min()} to {tps.trials.max()}")

**Result: reproduced.** The five sessions hold 184 to 398 trials, 290.2 ± 91.2 pooled and 272.5
over the three per-mouse means, both within 245 ± 71. Counting only trials with imaging gives
238.6 pooled and 225.2 over mice, also within. The test rests on 5 sessions of 3 mice; the paper's
number covers every imaging session of 8 mice, and how these five were chosen for deposit is not
documented, so they may not be a representative sample: the agreement is weak evidence. The trial
tables number trials from 1 without gaps, consistent with whole behavioural sessions.

## results-whisking-amplitude

> The task elicited larger amplitude whisker movements than similar two-position tasks, 55.5 ±
> 13.3 degrees (mean ± SD, n = 8 mice) against about 20 degrees.

Criterion: readings (a) per-trial range of whisker angle (maximum minus minimum) over the trial,
(b) the same over the sample epoch, (c) the peak Hilbert amplitude of whisking per trial; each
averaged per mouse, then over mice; reproduced when the mean over mice is within 55.5 ± 13.3 under
one reading.

Method details. The sample epoch is the pole-in-reach interval of the data set's `poleInReach`
series (the paper's "sample epoch (pole in reach)"). Reading (c) follows the Supplemental
Experimental Procedures ("the amplitude of whisking, obtained using the Hilbert transform (as
described previously; see (Huber et al., 2012))"); the band used to isolate whisking is not given,
and 6 to 30 Hz is assumed (helper docstring), with 4 to 30 and 8 to 25 Hz shown. Only trials the
data set lists as valid whisker trials are used.

In [ ]:
rows = []
for s in sessions:
    for band, w in s["whisking"].items():
        rows.append(pd.DataFrame({"mouse": s["meta"]["animalID"], "session": s["name"], "band": str(band),
                                  "range_trial": w["range_trial"], "range_sample": w["range_sample"],
                                  "hilbert_peak": w["hilbert_peak"]}))
wa = pd.concat(rows)
main = wa[wa.band == "(6.0, 30.0)"]
print("trials per mouse:", main.groupby("mouse").size().to_dict())
per_session = main.groupby(["mouse", "session"])[["range_trial", "range_sample", "hilbert_peak"]].mean()
print("\nper session (mean over trials, degrees):")
print(per_session.round(1).to_string())
per_mouse = main.groupby("mouse")[["range_trial", "range_sample", "hilbert_peak"]].mean()
print("\nper mouse (mean over trials, degrees):")
print(per_mouse.round(1).to_string())
lo, hi = 55.5 - 13.3, 55.5 + 13.3
print(f"\nmean over {len(per_mouse)} mice, against [{lo:.1f}, {hi:.1f}]:")
for k, name in [("range_trial", "(a) range over the trial"), ("range_sample", "(b) range over the sample epoch"),
                ("hilbert_peak", "(c) peak Hilbert amplitude, 6-30 Hz")]:
    m = per_mouse[k].mean()
    print(f"  {name}: {m:.1f} +/- {per_mouse[k].std():.1f} (SD over mice); within: {lo <= m <= hi}")
print("\nreading (c) with other bands (mean over mice):")
for band, g in wa.groupby("band"):
    print(f"  {band} Hz: {g.groupby('mouse').hilbert_peak.mean().mean():.1f}")
print("\nper-trial ranges of each reading (all mice, 6-30 Hz): ")
for k in ["range_trial", "range_sample", "hilbert_peak"]:
    print(f"  {k}: min {main[k].min():.1f}, 5/50/95 pct {np.nanpercentile(main[k], [5, 50, 95]).round(1).tolist()}, "
          f"max {main[k].max():.1f}, missing {int(main[k].isna().sum())}")

In [ ]:
# Variants, not readings the criterion lists: (a) with the 1st to 99th percentile range, which
# removes the rare +/-90 degree samples, and twice the peak Hilbert amplitude (a peak-to-peak
# measure, since the Hilbert amplitude is half the excursion of a sinusoid).
rob = pd.DataFrame([{"mouse": s["meta"]["animalID"], "r": r} for s in sessions for r in s["whisking_robust"]])
rob_m = rob.groupby("mouse").r.mean()
print("(a) with 1-99 percentile range, per mouse:", rob_m.round(1).to_dict(), f"; mean over mice {rob_m.mean():.1f}")
pp = 2 * per_mouse["hilbert_peak"]
print("2 x peak Hilbert amplitude (6-30 Hz), per mouse:", pp.round(1).to_dict(), f"; mean over mice {pp.mean():.1f} +/- {pp.std():.1f}")
print(f"within [{lo:.1f}, {hi:.1f}]: robust (a) {lo <= rob_m.mean() <= hi}; 2 x Hilbert {lo <= pp.mean() <= hi}")

In [ ]:
# Tracking errors (round-1 verifier, problems 1 and 2). Reading (a) without the samples near the
# tracker's +/-90 degree limit (|angle| > 85), without one- or two-sample spikes (more than 20
# degrees from the 5-sample running median), with other thresholds, and after unwrapping angles
# that pass 90 degrees; reading (c) without 0.2 s at each end of the trial, where filtering and
# the Hilbert transform inflate the amplitude (checked first on a sinusoid), and the same after
# the artefact samples are interpolated. Thresholds are standard choices (helper docstring).
full, trim = ssc1.hilbert_peak_synthetic(20.0)
print(f"synthetic 10 Hz sinusoid, amplitude 20: peak Hilbert amplitude {full:.1f} over the whole 4 s, "
      f"{trim:.1f} without 0.2 s at each end\n")
allw = pd.concat([pd.DataFrame({"mouse": s["meta"]["animalID"],
                                **{k: v for k, v in s["whisking"][(6.0, 30.0)].items() if k != "fs"}})
                  for s in sessions])
cnt = allw.groupby("mouse").agg(trials=("trial", "size"),
                                with_abs_over_85=("n_limit", lambda n: int((n > 0).sum())),
                                with_spike=("n_spike", lambda n: int((n > 0).sum())),
                                with_wrap=("n_wrap", lambda n: int((n > 0).sum())))
print("trials per mouse with at least one artefact sample:")
print(cnt.to_string())
variants = [("range_trial", "(a) as listed"), ("range_trial_limit", "(a) without |angle| > 85"),
            ("range_trial_spike", "(a) without spikes > 20 deg"),
            ("range_trial_unwrap_spike", "(a) unwrapped, without spikes"),
            ("range_trial_limit70", "(a) without |angle| > 70"), ("range_trial_limit80", "(a) without |angle| > 80"),
            ("range_trial_limit88", "(a) without |angle| > 88"), ("range_trial_spike10", "(a) without spikes > 10 deg"),
            ("range_trial_spike40", "(a) without spikes > 40 deg"),
            ("range_sample", "(b) as listed"), ("range_sample_limit", "(b) without |angle| > 85"),
            ("range_sample_spike", "(b) without spikes > 20 deg"),
            ("hilbert_peak", "(c) as listed"), ("hilbert_peak_trim", "(c) without 0.2 s at each end"),
            ("hilbert_peak_trim_clean", "(c) without ends, artefacts interpolated")]
mean_m = allw.groupby("mouse")[[k for k, _ in variants]].mean()
med_m = allw.groupby("mouse")[[k for k, _ in variants]].median()
rows = []
for k, name in variants:
    rows.append({"measure": name, **{m: round(mean_m.loc[m, k], 1) for m in mean_m.index},
                 "mean over mice": round(mean_m[k].mean(), 1), "SD over mice": round(mean_m[k].std(), 1),
                 "within": bool(lo <= mean_m[k].mean() <= hi),
                 "median-per-mouse mean": round(med_m[k].mean(), 1),
                 "within (medians)": bool(lo <= med_m[k].mean() <= hi)})
print(f"\nper-mouse means of each measure (degrees), and the mean over mice against [{lo:.1f}, {hi:.1f}]:")
print(pd.DataFrame(rows).to_string(index=False))
print("\nper-mouse medians over trials (degrees):")
print(med_m[["range_trial", "range_trial_spike", "range_sample", "hilbert_peak"]].round(1).to_string())
print("\nangle samples around the first two steps of more than 90 degrees, per session:")
for s in sessions:
    for ex in s["whisker_summary"]["jump_examples"]:
        print(f"  {s['name']}: {ex}")
pp2 = 2 * mean_m["hilbert_peak_trim_clean"]
print(f"\n2 x peak Hilbert amplitude without ends, artefacts interpolated: per mouse {pp2.round(1).to_dict()}, "
      f"mean over mice {pp2.mean():.1f} +/- {pp2.std():.1f}; within {lo <= pp2.mean() <= hi}")


In [ ]:
# Multi-sample tracking errors (round-2 verifier, point 1). The 5-sample spike mask misses jumps
# that last three or more samples. Two masks for them: samples more than 20 degrees from a running
# median of 11 or 21 samples (jumps of up to 5 or 10 samples), and the jump mask of the helper
# (a step of more than 30 degrees in one sample, back within 25 samples, with other values). The
# step sizes show which sessions hold such errors: the 99th percentile step of a smooth whisk.
print("step between successive angle samples (degrees per 2 ms), percentiles 50, 99, 99.9, 99.99:")
for s in sessions:
    print(f"  {s['name']}: {np.round(s['whisker_summary']['step_pct'], 1).tolist()}")
cnt2 = allw.groupby("mouse").agg(trials=("trial", "size"),
                                 with_jump=("n_jump", lambda n: int((n > 0).sum())),
                                 with_med21=("n_med", lambda n: int((n > 0).sum())))
print("\ntrials per mouse with samples in the jump mask (30 degrees, 25 samples) and the 21-sample median mask:")
print(cnt2.to_string())
mvars = [("range_trial", "(a) as listed"), ("range_trial_limit", "(a) without |angle| > 85"),
         ("range_trial_spike", "(a) without spikes (5-sample median)"),
         ("range_trial_med11", "(a) without jumps, 11-sample median"),
         ("range_trial_med21", "(a) without jumps, 21-sample median"),
         ("range_trial_jump20_10", "(a) jump mask, 20 deg, 10 samples"),
         ("range_trial_jump20_25", "(a) jump mask, 20 deg, 25 samples"),
         ("range_trial_jump30_10", "(a) jump mask, 30 deg, 10 samples"),
         ("range_trial_jump30_25", "(a) jump mask, 30 deg, 25 samples"),
         ("range_trial_jump30_50", "(a) jump mask, 30 deg, 50 samples"),
         ("range_trial_jump50_25", "(a) jump mask, 50 deg, 25 samples"),
         ("range_trial_lsj", "(a) without limit, spike and jump masks"),
         ("range_trial_all", "(a) without every mask, 21-sample median too")]
mm = allw.groupby("mouse")[[k for k, _ in mvars]].mean()
md = allw.groupby("mouse")[[k for k, _ in mvars]].median()
rows = [{"measure": name, **{m: round(mm.loc[m, k], 1) for m in mm.index},
         "mean over mice": round(mm[k].mean(), 2), "SD over mice": round(mm[k].std(), 1),
         "within": bool(lo <= mm[k].mean() <= hi), "above bound by": round(mm[k].mean() - hi, 2),
         "median-per-mouse mean": round(md[k].mean(), 1)} for k, name in mvars]
print(f"\nreading (a) per mouse and over mice (degrees), against [{lo:.1f}, {hi:.1f}]:")
print(pd.DataFrame(rows).to_string(index=False))
# Trials with no sample in any mask. A selection, not a cleaning: it shows the range of the trials
# the tracker followed throughout, and it may favour smaller sweeps (compare with all trials).
cl = allw[(allw.n_limit == 0) & (allw.n_spike == 0) & (allw.n_jump == 0) & (allw.n_med == 0)]
g = cl.groupby("mouse").range_trial
print("\ntrials with no sample in any mask: n", g.size().to_dict(), "; mean range", g.mean().round(1).to_dict(),
      f"; mean over mice {g.mean().mean():.1f}; all trials", mm["range_trial"].round(1).to_dict())

**Result: not reproduced.** Over the three mice in the data set, the mean whisker angle range per
trial is 74.7 degrees over the whole trial (a), 29.0 over the sample epoch (b), and the mean peak
Hilbert amplitude is 26.9 (c). None is within 55.5 ± 13.3 (42.2 to 68.8). Reading (c) gives 22.2 to
30.4 with the other bands and stays outside.

Removing the tracking errors brings reading (a) close to the upper bound but, with every mask that
removes only errors, not inside it. Samples above 85 degrees occur in 2 to 52 trials per mouse,
one- or two-sample spikes in 5 to 152 and wraps past 90 degrees in 3 to 43, most in an229717 (52,
152 and 43 of its 183 trials). Without them reading (a) is 72.4 (no samples above 85 degrees) or
71.9 (no spikes). It falls steadily as the threshold on |angle| drops: 74.1 at 88 degrees, 72.4 at
85, 70.3 at 80 and 67.4 at 70, the last two also removing the tops of real protractions. With the
spike threshold at 10 to 40 degrees it is 71.7 to 72.1. Unwrapping angles past 90 degrees raises it
to 77.0, as some protractions pass 90 degrees and wrap to the other sign (for example 84.6, 89.2,
-87.5, -88.7, 89.3, 83.9 in an197522); other errors are single samples at the limit (87.3, 37.8,
-89.2, 41.7 in an229717) or jumps that last three samples or more (-24.1, 72.8, 72.8, 72.1, -23.1
in an229719), which the 5-sample spike filter does not remove.

Jumps of several samples are nearly all in an229717: its 99.9th percentile step between samples is
34.5 degrees against 4.5 to 7.0 in the other sessions, and 116 of its 183 trials hold a step over 30
degrees that returns within 25 samples, against 9 of 764 and 4 of 496 in the other mice. Removing
these jumps gives reading (a) 69.7 to 70.1 with the six step and duration values tried (69.8 at 30
degrees and 25 samples), and 69.2 with the limit and spike masks as well, 0.4 above the bound; an229717
falls from 99.2 to 84.0 to 86.0, and the other mice by at most 1.0. Only the 21-sample running median,
alone (68.6) or with every other mask (68.3), brings it inside; that mask also flags samples in 79 of
764 trials of an197522, whose largest steps (99.99th percentile 9.1 to 11.3 degrees) show no jumps,
and lowers that mouse from 70.1 to 69.0, so it also removes the peaks of real whisks. The trials
with no sample in any mask (679, 24 and 473 per mouse) give 59.4 over mice, but they are a selection:
in an197522 they average 66.1 against 70.6 for all its trials. So the best-supported cleaning leaves
reading (a) 0.4 to 1.3 above the bound, and whether a cleaned reading (a) falls inside depends on a
choice the paper does not describe. The outcome stays not reproduced, by a narrow margin.

The filter's edge effect inflates the peak Hilbert amplitude of a sinusoid of amplitude 20 to 22.0,
and 20.1 once 0.2 s at each end is left out; on the data, leaving out the ends changes reading (c)
from 26.9 to 26.6, so the peaks lie inside the trials. With the multi-sample jumps removed an229717
still has a mean range of 84.0 to 86.0, the largest of the three mice, and its median of 87.2 is
unchanged by the 1- or 2-sample spikes.

Measures the criterion does not list come within the tolerance: the mean over mice of the per-mouse
medians of reading (a) (68.6, 0.2 below the upper bound, and 68.3 to 68.6 under every mask), the
trial range between the 1st and 99th percentiles of angle (49.2), and twice the peak Hilbert
amplitude, a peak-to-peak measure (53.7 ± 15.7, or 53.2 without the ends and artefacts). The part
of the statement the criterion does not test, that the movements are larger than about 20 degrees,
holds for every full-sweep measure (28.6 to 77.0). The paper does not define the measure, and the
test rests on 3 of the 8 mice.

## Summary

| Claim | Outcome |
|---|---|
| results-trials-per-session | reproduced (290 pooled, 273 over mice; 245 ± 71) |
| results-whisking-amplitude | not reproduced (74.7, 29.0, 26.9 degrees under the three readings; reading (a) 69.2 without every kind of tracking error, 0.4 above the bound; 55.5 ± 13.3) |
| table-s1-roi-counts | blocked, data missing (3 of 8 mice, 1 or 2 sessions each) |
| results-gaba-fraction, fig4d-gaba-reward-cue, fig2e-fraction-imaged, fig1e-days-to-criterion, fig1g-early-stereotypy, supp-trim-all-chance, figs4-dendrite-soma | blocked, data missing |
| 18 claims that need layer, cell-class or column labels (none of the data set's identifier or description fields carries one) | blocked, metadata insufficient |
| neuropil, L4 axon, learning and anesthetized claims (14) | blocked, data missing |
| methods-event-detection | blocked, methods underspecified |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

Differences from the paper's method. Both tested claims rest on 3 of the paper's 8 mice and
5 of its imaging sessions. The whisking amplitude measure is not defined in the paper; the three
readings of the claims file are shown, with a band for the Hilbert reading assumed (6 to 30 Hz),
and each is shown again without the tracking errors in the whisker angle.

What was found about the data set.
- It holds 5 sessions of 3 of the 8 volume-imaging mice (20% to 72% of their Table S1 ROIs), and
  none of the learning, L4 axon, anesthetized, L5 soma and dendrite or training data.
- The description lists raw fluorescence and extracted calcium events for each ROI, and
  multi-session `vol_xxx` files; none is in the data set.
- No ROI carries a layer, cell-class or column label, and no plane carries a depth or position
  relative to the others, so the paper's comparisons between classes and columns cannot be made.
- One subvolume (an197522, 2013-03-08, `fov_04001`) is a single plane at 29 Hz, which the
  description does not mention; many subvolumes have ROIs on fewer than three planes.
- The event series structure is named `eventSeriesArrayHash`, not `eventSeriesHash`; one file
  name is misspelt (`sesssion.mat`).
- In an197522 on 2013-03-10 the dF/F response to touch could not be shown to share the trial clock;
  any dF/F analysis of that session would inherit this.
- The whisker angle holds tracking errors: samples jumping to the tracker's +/-90 degree limit for
  one or two samples, jumps to a distant value lasting three samples or more (mostly in an229717,
  in 116 of its 183 trials), and protractions past 90 degrees that wrap to the other sign. None is
  described.

In [ ]:
print(f"notebook ran in {time.time() - started:.0f} s; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")